In [1]:
'''
changed dropout and loss fucntions
dropout 0.2
loss fn: 
1. first 20: CrossEntropyLoss
2. 21+ : FocalLoss

fixing Bottleneck Expansion in FeatureReduce

chnaged image input dimension to 224

changing batch size to 64

n_qubit =10

removing randomrotation from training


'''

'\nchanged dropout and loss fucntions\ndropout 0.2\nloss fn: \n1. first 20: CrossEntropyLoss\n2. 21+ : FocalLoss\n\nfixing Bottleneck Expansion in FeatureReduce\n'

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np
import random
import os
from torchvision.datasets import ImageFolder
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight

# ========== SEEDING ==========
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)

# ========== DEVICE ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== CONFIGURATION ==========
n_qubits = 10
batch_size = 64
num_classes = 26
num_epochs = 70
lr = 0.0005

# ========== DATA TRANSFORMS ==========
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    #transforms.RandomRotation(10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ========== DATASET LOADING ==========
TRAIN_PATH = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/train'
VAL_PATH   = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/val'
TEST_PATH  = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH, transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH, transform=eval_transform)
    print("** Dataset loaded successfully **")
    
except Exception as e:
    print(f"Error loading datasets: {e}")
    # Fallback for testing logic if paths don't exist
    os.makedirs("dummy_data/train/class1", exist_ok=True)
    os.makedirs("dummy_data/val/class1", exist_ok=True)
    os.makedirs("dummy_data/test/class1", exist_ok=True)
    train_dataset = ImageFolder("dummy_data/train", transform=train_transform)
    val_dataset = ImageFolder("dummy_data/val", transform=eval_transform)
    test_dataset = ImageFolder("dummy_data/test", transform=eval_transform)

# ========== CLASS WEIGHTS ==========

try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(class_weight='balanced',
                                         classes=np.unique(labels),
                                         y=labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print(f"Class weights calculated: {class_weights_tensor[:5]}... (showing first 5)")
    
except:
    print("Warning: Could not calculate class weights. Using ones.")
    class_weights_tensor = torch.ones(num_classes).to(device)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# ========== QUANTUM CIRCUIT ==========
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Angle Embedding
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    
    # Basic Entangler Layers
    for l in range(weights.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (6, n_qubits)}

# ========== MODEL ARCHITECTURE ==========
class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),    # 128 -> 64
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv2d(8, 16, 3, stride=2, padding=1),   # 64 -> 32
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # 32 -> 16
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # 16 -> 8
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv2d(64, 128, 3, stride=2, padding=1), # 8 -> 4
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 224, 3, stride=2, padding=1), # 8 -> 4
            nn.BatchNorm2d(224),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))                # 4x4 -> 1x1
        )
        self.fc_expand = nn.Linear(224, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc_expand(x))
        return self.fc_project(x)

class HybridQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)
        
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.tanh(x)
        q_out = self.q_layer(x)
        return self.classifier(q_out)

# ========== DYNAMIC FOCAL LOSS ==========
class ScheduledFocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=0, reduction='mean'):
        super(ScheduledFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma # Init with 0 for pure CrossEntropy behavior
        self.reduction = reduction

    def forward(self, inputs, targets):
        # We always use the calculated class weights
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=class_weights_tensor)
        pt = torch.exp(-ce_loss)
        
        # When gamma=0, this term becomes 1, leaving just Weighted CE.
        # When gamma>0, this term down-weights easy examples.
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ========== TRAINING FUNCTIONS ==========
def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct = 0.0, 0
    
    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()
    
    return total_loss / len(dataloader), correct / len(dataloader.dataset)

def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return total_loss / len(dataloader), correct / total

# ========== MAIN EXECUTION ==========
print("Initializing Model...")
model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)

# IMPLEMENTATION OF EXPERIMENT A:
# Start with gamma=0.0 (Pure Weighted Cross Entropy)
loss_fn = ScheduledFocalLoss(alpha=1, gamma=0.0)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

early_stopping_patience = 10
best_val_loss = float('inf')
epochs_without_improvement = 0

print(f"Starting Training for {num_epochs} epochs with Two-Stage Loss Schedule...")

for epoch in range(num_epochs):
    
    # ---------------------------------------------------------
    # EXPERIMENT A LOGIC: THE SWITCH
    # ---------------------------------------------------------
    if epoch < 20:
        loss_mode = "Stage 1: CrossEntropy (Gamma=0)"
        loss_fn.gamma = 0.0
    elif epoch == 20:
        print("\n" + "="*50)
        print(">>> SWITCHING TO STAGE 2: FOCAL LOSS (Gamma=1.0) <<<")
        print("="*50 + "\n")
        loss_mode = "Stage 2: FocalLoss (Gamma=1.0)"
        loss_fn.gamma = 1.0
    else:
        loss_mode = "Stage 2: FocalLoss (Gamma=1.0)"
        loss_fn.gamma = 1.0
    # ---------------------------------------------------------

    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} | {loss_mode}")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "exp-4_1_4.pth")
        print("   💾 Best model saved.")
    else:
        epochs_without_improvement += 1
        print(f"   🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"⏹️ Early stopping triggered after {epoch+1} epochs.")
        break

Using device: cuda
** Dataset loaded successfully **
Class weights calculated: tensor([1.0229, 1.0340, 1.0312, 1.0229, 1.0202], device='cuda:0')... (showing first 5)
Initializing Model...
Starting Training for 70 epochs with Two-Stage Loss Schedule...


Epoch 1/70 | Stage 1: CrossEntropy (Gamma=0)
   Train Loss: 3.1822 | Train Acc: 0.0735 | Val Acc: 0.0856
   💾 Best model saved.


Epoch 2/70 | Stage 1: CrossEntropy (Gamma=0)
   Train Loss: 2.6393 | Train Acc: 0.1454 | Val Acc: 0.1550
   💾 Best model saved.


Epoch 3/70 | Stage 1: CrossEntropy (Gamma=0)
   Train Loss: 2.1413 | Train Acc: 0.2648 | Val Acc: 0.3420
   💾 Best model saved.


Epoch 4/70 | Stage 1: CrossEntropy (Gamma=0)
   Train Loss: 1.8439 | Train Acc: 0.3518 | Val Acc: 0.4463
   💾 Best model saved.


...

Epoch 36/70 | Stage 2: FocalLoss (Gamma=1.0)
   Train Loss: 0.2056 | Train Acc: 0.8831 | Val Acc: 0.8050
   🕒 No improvement for 6 epoch(s).


Epoch 37/70 | Stage 2: FocalLoss (Gamma=1.0)
   Train Loss: 0.1819 | Train Acc: 0.8937 | Val Acc: 0.8325
   🕒 No improvement for 7 epoch(s).


Epoch 38/70 | Stage 2: FocalLoss (Gamma=1.0)
   Train Loss: 0.1760 | Train Acc: 0.9025 | Val Acc: 0.8185
   🕒 No improvement for 8 epoch(s).


Epoch 39/70 | Stage 2: FocalLoss (Gamma=1.0)
   Train Loss: 0.1698 | Train Acc: 0.9022 | Val Acc: 0.8413
   🕒 No improvement for 9 epoch(s).


Epoch 40/70 | Stage 2: FocalLoss (Gamma=1.0)
   Train Loss: 0.1639 | Train Acc: 0.9071 | Val Acc: 0.8488
   🕒 No improvement for 10 epoch(s).
⏹️ Early stopping triggered after 40 epochs.


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import pennylane as qml
from pennylane.qnn import TorchLayer
from sklearn.metrics import classification_report
import numpy as np

# 1. SETUP DEVICE & CONFIG
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

n_qubits = 10
num_classes = 26

# 2. RE-DEFINE QUANTUM LAYER (Required for HybridQNN)
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Angle Embedding
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    
    # Basic Entangler Layers
    for l in range(weights.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (6, n_qubits)}

# 3. RE-DEFINE MODEL ARCHITECTURE (Required to load weights)
class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),    
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),   
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), 
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 224, 3, stride=2, padding=1), 
            nn.BatchNorm2d(224),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                
        )
        self.fc_expand = nn.Linear(224, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc_expand(x))
        return self.fc_project(x)

class HybridQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)
        
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.tanh(x)
        q_out = self.q_layer(x)
        return self.classifier(q_out)

# 4. LOAD TEST DATA
TEST_PATH = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/test'

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224, 224)), # Ensuring size matches training
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

print("Loading Test Data...")
try:
    test_dataset = ImageFolder(TEST_PATH, transform=eval_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    class_names = test_dataset.classes
    print(f"Classes found: {len(class_names)}")
except Exception as e:
    print(f"Error loading data: {e}")

# 5. EVALUATION FUNCTIONS
def get_predictions(model, loader, device):
    model.eval()
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            
    return np.array(y_true), np.array(y_pred)

def evaluate_checkpoint(checkpoint_path, test_loader, device, class_names):
    print(f"\nEvaluating Checkpoint: {checkpoint_path}")
    
    # Initialize fresh model structure
    model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)
    
    # Load weights
    try:
        state_dict = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(state_dict)
        print("Weights loaded successfully.")
    except FileNotFoundError:
        print(f"Error: File '{checkpoint_path}' not found.")
        return

    # Run predictions
    print("Running predictions on Test Set...")
    y_true, y_pred = get_predictions(model, test_loader, device)
    
    # Print Report
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(
        y_true, 
        y_pred, 
        target_names=class_names, 
        digits=4
    ))

# 6. RUN THE EVALUATION
# Make sure the filename matches what you actually saved (e.g., exp-4_1_3.pth or exp-4_1_4.pth)
evaluate_checkpoint(
    checkpoint_path="exp-4_1_4.pth", 
    test_loader=test_loader, 
    device=device, 
    class_names=class_names
)

Using device: cuda
Loading Test Data...
Classes found: 26

Evaluating Checkpoint: exp-4_1_4.pth
Weights loaded successfully.
Running predictions on Test Set...

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Adposhel     1.0000    1.0000    1.0000        60
       Agent     0.6379    0.7400    0.6852        50
     Allaple     0.9412    0.9057    0.9231        53
   Amonetize     0.9833    0.9672    0.9752        61
      Androm     0.7176    0.9839    0.8299        62
     Autorun     0.8621    0.8197    0.8403        61
   BrowseFox     0.9286    0.8667    0.8966        60
      Dinwod     0.9385    0.9839    0.9606        62
        Elex     0.9688    1.0000    0.9841        62
      Expiro     0.4330    0.6667    0.5250        63
      Fasong     1.0000    1.0000    1.0000        62
     HackKMS     1.0000    1.0000    1.0000        62
        Hlux     1.0000    1.0000    1.0000        62
    Injector     0.6351    0.7833    0.7015        60
 Insta